# BEA parser walkthrough

This notebook shows how to parse the official BEA `SUPPLY-USE` workbook family with `mario.parse_bea(...)`.

## Where the files come from

Relevant official pages:

- [Industry Input-Output Accounts Data](https://www.bea.gov/industry/input-output-accounts-data)
- [Guide to the interactive industry input-output accounts tables](https://www.bea.gov/resources/guide-interactive-industry-input-output-accounts-tables)
- [Direct `SUPPLY-USE.zip` bundle](https://apps.bea.gov/industry/release/zip/SUPPLY-USE.zip)

MARIO currently supports only the `SUPPLY-USE` bundle. It does not yet parse:

- `MAKE-USE-IMPORTS (BEFORE REDEFINITIONS)`
- `TOTAL AND DOMESTIC REQUIREMENTS`

## Supported levels and years

The parser currently supports three aggregation levels:

- `summary`: `1997` to `2024`
- `sector`: `1997` to `2024`
- `detail`: `2007`, `2012`, `2017`

These ranges were verified directly on the current official workbook bundle.

## Accepted path inputs

`parse_bea(...)` accepts any of these:

- the official `SUPPLY-USE.zip`
- one extracted directory containing the BEA workbooks
- one workbook path inside that directory

In all cases you still provide:

- `year=`
- `level=`

In [ ]:
import mario

In [ ]:
db = mario.parse_bea(
    path="/path/to/SUPPLY-USE.zip",
    year=2024,
    level="summary",
    calc_all=False,
)

db

In [ ]:
db = mario.parse_bea(
    path="/path/to/SUPPLY-USE",
    year=2024,
    level="sector",
    calc_all=False,
)

In [ ]:
db = mario.parse_bea(
    path="/path/to/Supply_Detail.xlsx",
    year=2017,
    level="detail",
    calc_all=False,
)

## What the parser builds

MARIO reads the BEA bundle as a split-native `SUT`:

- `S` from the BEA `Supply` workbook
- `U` and `Yc` from the BEA `Use` workbook
- `Va` from the use-workbook footer rows
- `Vc` from the supply-workbook commodity-side columns for imports, CIF/FOB adjustments, trade and transport margins, and product taxes

This is therefore a mixed-valuation table:

- supply is read at basic prices
- use is read at purchaser prices

In [ ]:
base = db["baseline"]
for name in ["S", "U", "Yc", "Va", "Vc"]:
    print(name, base[name].shape)

## Caveats

- `table="SUT"` is the only supported mode right now.
- The parser does not yet expose the separate `MAKE-USE-IMPORTS (BEFORE REDEFINITIONS)` bundle.
- The parser does not treat `TOTAL AND DOMESTIC REQUIREMENTS` as raw database inputs.
- `detail` coverage is much narrower than `summary` and `sector` in the current official release.